In [ ]:
import os
import json
import time
from typing import List, Dict, Tuple

import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor
from decord import VideoReader, cpu


# =========================================================
# PATHS
# =========================================================
DATA_ROOT = "./MSVD"
CHECKPOINT = "./output_msvd_clip/best.pt"
MODEL_NAME = "openai/clip-vit-base-patch32"
NUM_FRAMES = 8
TEXT_BATCH_SIZE = 64
VIDEO_BATCH_SIZE = 16
MAX_TEXT_LEN = 32
OUTPUT_JSON = "./output_msvd_clip/clip_eval_results.json"


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sample_frame_indices(total_frames: int, num_frames: int) -> List[int]:
    if total_frames <= 0:
        raise ValueError("Video has no frames")
    return np.linspace(0, total_frames - 1, num_frames, dtype=int).tolist()


def load_video_frames(video_path: str, num_frames: int) -> List[Image.Image]:
    vr = VideoReader(video_path, ctx=cpu(0))
    idxs = sample_frame_indices(len(vr), num_frames)
    frames = vr.get_batch(idxs).asnumpy()
    return [Image.fromarray(frame).convert("RGB") for frame in frames]


def build_test_set(data_root: str) -> Tuple[List[Dict], List[Dict]]:
    test_file = os.path.join(data_root, "msvd_test.json")
    video_root = os.path.join(data_root, "raw_videos")

    if not os.path.exists(test_file):
        raise FileNotFoundError(f"Missing file: {test_file}")
    if not os.path.exists(video_root):
        raise FileNotFoundError(f"Missing folder: {video_root}")

    records = load_json(test_file)

    videos = {}
    queries = []

    for item in records:
        video_name = item["video"]
        video_id = item.get("video_id", video_name)
        video_path = os.path.join(video_root, video_name)

        if not os.path.exists(video_path):
            continue

        if video_id not in videos:
            videos[video_id] = {
                "video_id": video_id,
                "video_name": video_name,
                "video_path": video_path,
            }

        captions = item["caption"]
        if isinstance(captions, str):
            captions = [captions]

        for cap in captions:
            cap = " ".join(str(cap).strip().split())
            if cap:
                queries.append({
                    "video_id": video_id,
                    "caption": cap
                })

    return queries, list(videos.values())


def extract_tensor_features(output, kind="image"):
    """
    Make feature extraction robust across transformer versions / return types.
    """
    if torch.is_tensor(output):
        return output

    # Common CLIP projected embeddings
    if hasattr(output, "image_embeds") and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, "text_embeds") and output.text_embeds is not None:
        return output.text_embeds

    # Pooling outputs
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output

    # Last hidden state fallback
    if hasattr(output, "last_hidden_state") and output.last_hidden_state is not None:
        # take CLS token
        return output.last_hidden_state[:, 0, :]

    # Tuple/list fallback
    if isinstance(output, (tuple, list)) and len(output) > 0:
        if torch.is_tensor(output[0]):
            return output[0]

    raise TypeError(f"Could not extract tensor features from {type(output)}")


class CLIPVideoTextModel(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

    def encode_text(self, input_ids, attention_mask):
        # first try standard API
        try:
            text_features = self.clip.get_text_features(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
        except Exception:
            # fallback to raw forward
            out = self.clip.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            text_features = extract_tensor_features(out, kind="text")

        text_features = extract_tensor_features(text_features, kind="text")
        return F.normalize(text_features, dim=-1)

    def encode_video(self, pixel_values):
        # pixel_values: [B, T, 3, H, W]
        bsz, t, c, h, w = pixel_values.shape
        flat_pixels = pixel_values.reshape(bsz * t, c, h, w)

        # first try standard API
        try:
            frame_features = self.clip.get_image_features(pixel_values=flat_pixels)
        except Exception:
            # fallback to vision model directly
            out = self.clip.vision_model(pixel_values=flat_pixels)
            frame_features = extract_tensor_features(out, kind="image")

        frame_features = extract_tensor_features(frame_features, kind="image")
        frame_features = frame_features.reshape(bsz, t, -1)
        video_features = frame_features.mean(dim=1)

        return F.normalize(video_features, dim=-1)


def load_checkpoint(model: nn.Module, checkpoint_path: str):
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    ckpt = torch.load(checkpoint_path, map_location="cpu")

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"], strict=False)
    else:
        model.load_state_dict(ckpt, strict=False)

    print(f"Loaded checkpoint from: {checkpoint_path}")


@torch.no_grad()
def encode_videos(model, processor, videos, device, num_frames, batch_size):
    all_embeds = []
    video_ids = []

    for start in tqdm(range(0, len(videos), batch_size), desc="Encoding videos"):
        batch = videos[start:start + batch_size]

        frames_nested = [load_video_frames(v["video_path"], num_frames) for v in batch]
        flat_images = [img for frames in frames_nested for img in frames]

        image_inputs = processor(images=flat_images, return_tensors="pt")
        pixel_values = image_inputs["pixel_values"]

        pixel_values = pixel_values.reshape(
            len(batch), num_frames, *pixel_values.shape[1:]
        ).to(device)

        video_embeds = model.encode_video(pixel_values)
        all_embeds.append(video_embeds.cpu())
        video_ids.extend([v["video_id"] for v in batch])

    return torch.cat(all_embeds, dim=0), video_ids


@torch.no_grad()
def encode_queries(model, processor, queries, device, max_text_len, batch_size):
    all_embeds = []
    gt_video_ids = []

    for start in tqdm(range(0, len(queries), batch_size), desc="Encoding texts"):
        batch = queries[start:start + batch_size]
        texts = [q["caption"] for q in batch]
        gt_video_ids.extend([q["video_id"] for q in batch])

        text_inputs = processor(
            text=texts,
            padding=True,
            truncation=True,
            max_length=max_text_len,
            return_tensors="pt"
        )

        text_embeds = model.encode_text(
            text_inputs["input_ids"].to(device),
            text_inputs["attention_mask"].to(device)
        )
        all_embeds.append(text_embeds.cpu())

    return torch.cat(all_embeds, dim=0), gt_video_ids


def compute_metrics(similarity: np.ndarray, gt_video_ids: List[str], video_ids: List[str]) -> Dict[str, float]:
    video_id_to_idx = {vid: i for i, vid in enumerate(video_ids)}
    gt_indices = np.array([video_id_to_idx[v] for v in gt_video_ids], dtype=np.int64)

    sorted_idx = np.argsort(-similarity, axis=1)
    ranks = []

    for i in range(len(gt_indices)):
        rank = int(np.where(sorted_idx[i] == gt_indices[i])[0][0]) + 1
        ranks.append(rank)

    ranks = np.array(ranks)

    return {
        "R@1": float(np.mean(ranks <= 1)),
        "R@5": float(np.mean(ranks <= 5)),
        "R@10": float(np.mean(ranks <= 10)),
        "MRR": float(np.mean(1.0 / ranks)),
        "MeanRank": float(np.mean(ranks)),
        "MedianRank": float(np.median(ranks)),
        "Top1Accuracy": float(np.mean(ranks == 1)),
        "Top5Accuracy": float(np.mean(ranks <= 5)),
    }


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    print("Current working dir:", os.getcwd())
    print("DATA_ROOT exists:", os.path.exists(DATA_ROOT))
    print("CHECKPOINT exists:", os.path.exists(CHECKPOINT))

    queries, videos = build_test_set(DATA_ROOT)
    print(f"Queries: {len(queries)} | Videos: {len(videos)}")

    processor = CLIPProcessor.from_pretrained(MODEL_NAME)
    model = CLIPVideoTextModel(MODEL_NAME).to(device)
    load_checkpoint(model, CHECKPOINT)
    model.eval()

    start_time = time.time()

    video_embeds, video_ids = encode_videos(
        model, processor, videos, device, NUM_FRAMES, VIDEO_BATCH_SIZE
    )
    text_embeds, gt_video_ids = encode_queries(
        model, processor, queries, device, MAX_TEXT_LEN, TEXT_BATCH_SIZE
    )

    similarity = (text_embeds @ video_embeds.T).numpy()
    metrics = compute_metrics(similarity, gt_video_ids, video_ids)
    metrics["avg_query_latency_sec"] = float((time.time() - start_time) / max(len(queries), 1))

    print("\nCLIP Retrieval Metrics")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    print(f"\nSaved to {OUTPUT_JSON}")


main()

Using device: cuda
Current working dir: /app
DATA_ROOT exists: True
CHECKPOINT exists: True
Queries: 27763 | Videos: 670


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 13671.85it/s]


Loaded checkpoint from: ./output_msvd_clip/best.pt


Encoding texts: 100%|██████████| 434/434 [00:06<00:00, 69.39it/s]



CLIP Retrieval Metrics
R@1: 0.3664
R@5: 0.6792
R@10: 0.7867
MRR: 0.5067
MeanRank: 14.1342
MedianRank: 2.0000
Top1Accuracy: 0.3664
Top5Accuracy: 0.6792
avg_query_latency_sec: 0.0025

Saved to ./output_msvd_clip/clip_eval_results.json


In [ ]:

import json
import pandas as pd

RESULT_PATH = "./output_msvd_clip/clip_eval_results.json"

with open(RESULT_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print("Raw results:")
print(results)

df = pd.DataFrame(list(results.items()), columns=["Metric", "Value"])
df

Raw results:
{'R@1': 0.36638691784029104, 'R@5': 0.6792493606598711, 'R@10': 0.7866945214854303, 'MRR': 0.5066888414683666, 'MeanRank': 14.134207398335915, 'MedianRank': 2.0, 'Top1Accuracy': 0.36638691784029104, 'Top5Accuracy': 0.6792493606598711, 'avg_query_latency_sec': 0.002511474733517887}


,Metric,Value
0,R@1,0.366387
1,R@5,0.679249
2,R@10,0.786695
3,MRR,0.506689
4,MeanRank,14.134207
5,MedianRank,2.000000
6,Top1Accuracy,0.366387
7,Top5Accuracy,0.679249
8,avg_query_latency_sec,0.002511
